In [ ]:
# ============================================================
# CONFIG — edit only this cell
# ============================================================
from pathlib import Path
from mdatools.config import AnalysisConfig

cfg = AnalysisConfig(
    ligand_resname = "LIG",       # resname of your ligand in the topology
    topology_glob  = "*.pdb",
    trajectory_glob= "*.xtc",
    dt_ns          = 2.0,
)

REPLICA_ROOTS = [
    Path("../run01"),
    Path("../run02"),
]

# Residue IDs that define the binding pocket (for highlighting / sub-analysis)
POCKET_RESIDS = [50, 51, 74, 108, 112, 152, 184, 185, 186]

OUTPUT_DIR = Path("./figures")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# ============================================================

## Step 1 — Run RMSF analysis for all replicas

In [ ]:
from mdatools.analysis.rmsf import run_rmsf_batch

results = run_rmsf_batch(REPLICA_ROOTS, cfg)

# Quick summary
for name, res in results.items():
    print(f"{name}: {len(res.protein_df)} protein residues, {len(res.ligand_df)} ligand atoms")

## Step 2 — Protein per-residue RMSF (single replica)

In [ ]:
from mdatools.plotting.rmsf_plots import plot_protein_rmsf

first_result = next(iter(results.values()))

fig = plot_protein_rmsf(
    first_result,
    pocket_resids=POCKET_RESIDS,
    save_path=OUTPUT_DIR / f"protein_rmsf_{first_result.sample_name}.png",
)
fig

## Step 3 — Ligand per-atom RMSF

In [ ]:
from mdatools.plotting.rmsf_plots import plot_ligand_rmsf

fig = plot_ligand_rmsf(
    first_result,
    save_path=OUTPUT_DIR / f"ligand_rmsf_{first_result.sample_name}.png",
)
fig

## Step 4 — RMSF comparison across replicas

In [ ]:
from mdatools.plotting.rmsf_plots import plot_rmsf_comparison

fig = plot_rmsf_comparison(
    list(results.values()),
    pocket_resids=POCKET_RESIDS,
    save_path=OUTPUT_DIR / "rmsf_comparison.png",
)
fig

## Step 5 — Binding site focused RMSF

In [ ]:
from mdatools.analysis.rmsf import RMSFAnalyzer
from mdatools.io.loaders import discover_replicas
from mdatools.universe import load_and_align

analyzer = RMSFAnalyzer(cfg)
binding_site_results = {}

for rep in discover_replicas(REPLICA_ROOTS, cfg):
    u = load_and_align(rep["topology"], rep["trajectory"], cfg)
    binding_site_results[rep["name"]] = analyzer.run_binding_site(
        u, pocket_resids=POCKET_RESIDS, sample_name=rep["name"]
    )

# Show highest-fluctuation binding site residue per replica
import pandas as pd

for name, res in binding_site_results.items():
    top = res.protein_df.sort_values("rmsf", ascending=False).head(3)
    print(f"\n{name} — most mobile binding site residues:")
    print(top.to_string(index=False))